In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
import pandas as pd
df=pd.read_csv("/kaggle/input/datasets/arjunmahesh09999/news-2-for/FOR_NEWS_2.csv")
cols = [
    'Solar8000/RR',
    'Solar8000/PLETH_SPO2',
    'Solar8000/HR',
    'Solar8000/ART_SBP',
    
    'Solar8000/FIO2'
]

df = df.sort_values(['patient_id', 'time'])

df[cols] = df.groupby('patient_id')[cols].ffill().bfill()

In [4]:
import numpy as np
import pandas as pd
from collections import deque

# =========================================================
# 1) CONFIG
# =========================================================
PATIENT_COL = "patient_id"
TIME_COL = "time"

RR_COL = "Solar8000/RR"
SPO2_COL = "Solar8000/PLETH_SPO2"
HR_COL = "Solar8000/HR"
SBP_COL = "Solar8000/ART_SBP"
FIO2_COL = "Solar8000/FIO2"

# No direct oxygen flag in your current CSV
OXY_FLAG_COL = None

# Fill missing values within each patient before scoring
DO_FFILL = True

# RR smoothing
RR_SMOOTH_WINDOW = 5

# FSM parameters
WINDOW_SIZE = 15
CONFIRM_LEN = 15
EMERGENCY_UPGRADE_COUNT = 10
NORMAL_DOWNGRADE_COUNT = 12


# =========================================================
# 2) RR SMOOTHING
# =========================================================
def smooth_rr_per_patient(df):
    """
    Smooth RR per patient using a 5-row rolling mean.
    First 4 rows are handled by mean of available rows only.
    """
    df = df.copy()
    df["RR_raw"] = df[RR_COL]

    df[RR_COL] = (
        df.groupby(PATIENT_COL)[RR_COL]
          .transform(lambda s: s.rolling(window=RR_SMOOTH_WINDOW, min_periods=1).mean())
    )

    return df


# =========================================================
# 3) SUBSCORES
# =========================================================
def score_rr(rr):
    if pd.isna(rr):
        return np.nan
    rr = float(rr)
    if rr <= 8:
        return 3
    elif rr <= 11:
        return 1
    elif rr <= 20:
        return 0
    elif rr <= 24:
        return 2
    else:
        return 3

def score_spo2_scale1(spo2):
    if pd.isna(spo2):
        return np.nan
    spo2 = float(spo2)
    if spo2 <= 91:
        return 3
    elif spo2 <= 93:
        return 2
    elif spo2 <= 95:
        return 1
    else:
        return 0

def score_hr(hr):
    if pd.isna(hr):
        return np.nan
    hr = float(hr)
    if hr <= 40:
        return 3
    elif hr <= 50:
        return 1
    elif hr <= 90:
        return 0
    elif hr <= 110:
        return 1
    elif hr <= 130:
        return 2
    else:
        return 3

def score_sbp(sbp):
    if pd.isna(sbp):
        return np.nan
    sbp = float(sbp)
    if sbp <= 90:
        return 3
    elif sbp <= 100:
        return 2
    elif sbp <= 110:
        return 1
    elif sbp <= 219:
        return 0
    else:
        return 3

def oxygen_score(row):
    """
    Official NEWS2 adds 2 points for supplemental oxygen.
    If you do not have a direct oxygen flag, FiO2 > 21 is used as a proxy.
    """
    if OXY_FLAG_COL is not None and OXY_FLAG_COL in row and not pd.isna(row[OXY_FLAG_COL]):
        try:
            return 2 if int(row[OXY_FLAG_COL]) == 1 else 0
        except Exception:
            return 0

    if FIO2_COL is not None and FIO2_COL in row and not pd.isna(row[FIO2_COL]):
        try:
            return 2 if float(row[FIO2_COL]) > 21 else 0
        except Exception:
            return 0

    return 0


# =========================================================
# 4) MODIFIED NEWS2 TOTAL
# =========================================================
def compute_modified_news2_row(row):
    s_rr = score_rr(row[RR_COL])
    s_spo2 = score_spo2_scale1(row[SPO2_COL])
    s_hr = score_hr(row[HR_COL])
    s_sbp = score_sbp(row[SBP_COL])
    s_o2 = oxygen_score(row)

    if any(pd.isna(x) for x in [s_rr, s_spo2, s_hr, s_sbp]):
        return pd.Series({
            "s_rr": np.nan,
            "s_spo2": np.nan,
            "s_hr": np.nan,
            "s_sbp": np.nan,
            "s_oxygen": s_o2,
            "news2_total": np.nan
        })

    total = int(s_rr + s_spo2 + s_hr + s_sbp + s_o2)

    return pd.Series({
        "s_rr": int(s_rr),
        "s_spo2": int(s_spo2),
        "s_hr": int(s_hr),
        "s_sbp": int(s_sbp),
        "s_oxygen": int(s_o2),
        "news2_total": total
    })

def severity_from_modified_news2(row):
    """
    0 = Normal
    1 = Critical
    2 = Emergency
    """
    score = row["news2_total"]
    if pd.isna(score):
        return np.nan

    any_red = any(x == 3 for x in [row["s_rr"], row["s_spo2"], row["s_hr"], row["s_sbp"]])

    if score >= 7 or any_red:
        return 2
    elif score >= 5:
        return 1
    else:
        return 0


# =========================================================
# 5) FSM
# =========================================================
def hierarchical_fsm_numeric(severity_labels):
    prev_confirmed = 0
    result = []

    window = deque(maxlen=WINDOW_SIZE)
    consec = {2: 0, 1: 0, 0: 0}

    for lab in severity_labels:
        if pd.isna(lab):
            result.append(prev_confirmed)
            continue

        lab = int(lab)

        for k in consec:
            consec[k] = consec[k] + 1 if k == lab else 0

        window.append(lab)

        # HARD CONFIRMATION
        if consec[2] >= CONFIRM_LEN:
            prev_confirmed = 2
            result.append(prev_confirmed)
            continue

        if consec[1] >= CONFIRM_LEN:
            prev_confirmed = 1
            result.append(prev_confirmed)
            continue

        if consec[0] >= CONFIRM_LEN:
            # Emergency -> Normal must pass via Critical
            if prev_confirmed == 2:
                prev_confirmed = 1
            else:
                prev_confirmed = 0
            result.append(prev_confirmed)
            continue

        # WINDOW LOGIC
        if len(window) == WINDOW_SIZE:
            cntN = window.count(0)
            cntC = window.count(1)
            cntE = window.count(2)

            if prev_confirmed == 2:
                if cntE == 0:
                    prev_confirmed = 1
                    result.append(prev_confirmed)
                    continue

            elif prev_confirmed == 1:
                if cntC == 0:
                    if cntE >= EMERGENCY_UPGRADE_COUNT:
                        prev_confirmed = 2
                        result.append(prev_confirmed)
                        continue
                    if cntN >= NORMAL_DOWNGRADE_COUNT:
                        prev_confirmed = 0
                        result.append(prev_confirmed)
                        continue

            elif prev_confirmed == 0:
                if cntN == 0:
                    prev_confirmed = 1
                    result.append(prev_confirmed)
                    continue

        result.append(prev_confirmed)

    return result


# =========================================================
# 6) FULL PIPELINE
# =========================================================
def build_modified_news2_and_fsm(df):
    df = df.copy()

    # Basic cleanup
    needed_cols = [PATIENT_COL, TIME_COL, RR_COL, SPO2_COL, HR_COL, SBP_COL]
    missing = [c for c in needed_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Sort
    df = df.sort_values([PATIENT_COL, TIME_COL]).reset_index(drop=True)

    # Convert to numeric
    for col in [RR_COL, SPO2_COL, HR_COL, SBP_COL]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    if FIO2_COL in df.columns:
        df[FIO2_COL] = pd.to_numeric(df[FIO2_COL], errors="coerce")

    # Optional forward-fill within each patient
    if DO_FFILL:
        fill_cols = [RR_COL, SPO2_COL, HR_COL, SBP_COL]
        if FIO2_COL in df.columns:
            fill_cols.append(FIO2_COL)
        if OXY_FLAG_COL is not None and OXY_FLAG_COL in df.columns:
            fill_cols.append(OXY_FLAG_COL)

        df[fill_cols] = df.groupby(PATIENT_COL)[fill_cols].ffill().bfill()

    # RR smoothing before scoring
    df = smooth_rr_per_patient(df)

    # Compute scores
    subscore_df = df.apply(compute_modified_news2_row, axis=1)
    df = pd.concat([df, subscore_df], axis=1)

    # Raw severity label
    df["severity_label"] = df.apply(severity_from_modified_news2, axis=1).astype("Int64")

    # FSM result label per patient
    result_parts = []
    for pid, g in df.groupby(PATIENT_COL, sort=False):
        g = g.copy()
        g["result_label"] = hierarchical_fsm_numeric(g["severity_label"].tolist())
        result_parts.append(g)

    df = pd.concat(result_parts, axis=0).sort_values([PATIENT_COL, TIME_COL]).reset_index(drop=True)
    df["result_label"] = df["result_label"].astype("Int64")

    # Readable classes
    label_map = {0: "Normal", 1: "Critical", 2: "Emergency"}
    df["severity_class"] = df["severity_label"].map(label_map)
    df["result_class"] = df["result_label"].map(label_map)

    return df


# =========================================================
# 7) RUN IT
# =========================================================
final_df = build_modified_news2_and_fsm(df)

print(final_df[
    [PATIENT_COL, TIME_COL, "RR_raw", RR_COL, SPO2_COL, HR_COL, SBP_COL,
     "news2_total", "severity_label", "result_label",
     "severity_class", "result_class"]
].head(20))

    patient_id  time  RR_raw  Solar8000/RR  Solar8000/PLETH_SPO2  \
0           64   0.0    11.0         11.00                  97.0   
1           64   2.0    11.0         11.00                  97.0   
2           64   4.0    11.0         11.00                  97.0   
3           64   6.0    10.0         10.75                  98.0   
4           64   8.0    10.0         10.60                  98.0   
5           64  10.0    10.0         10.40                  97.0   
6           64  12.0    13.0         10.80                  97.0   
7           64  14.0    15.0         11.60                  97.0   
8           64  16.0    15.0         12.60                  97.0   
9           64  18.0    18.0         14.20                  97.0   
10          64  20.0    18.0         15.80                  97.0   
11          64  22.0    18.0         16.80                  97.0   
12          64  24.0    16.0         17.00                  97.0   
13          64  26.0    16.0         17.20      

In [5]:
final_df.head()

,patient_id,time,Solar8000/ART_SBP,Solar8000/FIO2,Solar8000/RR,Solar8000/HR,Solar8000/PLETH_SPO2,RR_raw,s_rr,s_spo2,s_hr,s_sbp,s_oxygen,news2_total,severity_label,result_label,severity_class,result_class
0,64,0.0,-20.0,54.0,11.00,51.0,97.0,11.0,1,0,0,3,2,6,2,0,Emergency,Normal
1,64,2.0,-20.0,54.0,11.00,51.0,97.0,11.0,1,0,0,3,2,6,2,0,Emergency,Normal
2,64,4.0,-20.0,54.0,11.00,51.0,97.0,11.0,1,0,0,3,2,6,2,0,Emergency,Normal
3,64,6.0,-20.0,54.0,10.75,51.0,98.0,10.0,1,0,0,3,2,6,2,0,Emergency,Normal
4,64,8.0,-20.0,54.0,10.60,51.0,98.0,10.0,1,0,0,3,2,6,2,0,Emergency,Normal


In [9]:
final_df.to_csv("mNEWS_score.csv", index=False)

In [10]:
import pandas as pd
import os

def compare_medical_csvs(file1_path, file2_path):
    # Check if files exist before trying to read them
    if not os.path.exists(file1_path) or not os.path.exists(file2_path):
        print(f"Error: One or both files not found.\nFile 1: {file1_path}\nFile 2: {file2_path}")
        return

    # 1. Load the CSV files using the provided paths
    df1 = pd.read_csv(file1_path)
    df2 = pd.read_csv(file2_path)

    # 2. Merge dataframes on common identifiers
    # This acts as an inner join: only rows with same patient_id and time in both are kept
    merged_df = pd.merge(
        df1, 
        df2, 
        on=['patient_id', 'time'], 
        suffixes=('_csv1', '_csv2')
    )

    total_common_rows = len(merged_df)

    if total_common_rows == 0:
        print("No common rows found based on patient_id and time.")
        return

    # 3. Compare 'severity_label'
    # This creates a boolean mask where labels match and sums the 'True' values
    severity_matches = (merged_df['severity_label_csv1'] == merged_df['severity_label_csv2']).sum()
    severity_percentage = (severity_matches / total_common_rows) * 100

    # 4. Compare 'result_label'
    result_matches = (merged_df['result_label_csv1'] == merged_df['result_label_csv2']).sum()
    result_percentage = (result_matches / total_common_rows) * 100

    # 5. Combined match (both columns must match in the same row)
    combined_matches = (
        (merged_df['severity_label_csv1'] == merged_df['severity_label_csv2']) & 
        (merged_df['result_label_csv1'] == merged_df['result_label_csv2'])
    ).sum()
    combined_percentage = (combined_matches / total_common_rows) * 100

    # Output results
    print(f"--- Comparison Results ---")
    print(f"Total common rows found: {total_common_rows}")
    print(f"Severity Match:      {severity_matches} rows ({severity_percentage:.2f}%)")
    print(f"Result Match:        {result_matches} rows ({result_percentage:.2f}%)")
    print(f"Exact Match (Both):  {combined_matches} rows ({combined_percentage:.2f}%)")

# --- EXECUTION PART ---
# Define your file paths here
path1 = "/kaggle/working/mNEWS_score.csv"
path2 = "/kaggle/input/datasets/arjunmahesh09999/new-masterdata/MASTERDATA.csv"

# CALL the function to see the results
compare_medical_csvs(path1, path2)

--- Comparison Results ---
Total common rows found: 54750
Severity Match:      33612 rows (61.39%)
Result Match:        33323 rows (60.86%)
Exact Match (Both):  29618 rows (54.10%)
